# 01 - Bangun Dataset dari BDD100K

Mengubah BDD100K (format deteksi) menjadi dataset klasifikasi jenis kendaraan:
crop bbox kendaraan, peta kategori -> kelas, kondisi pencahayaan asli dari `timeofday`,
dan split train/val/test berbasis grup video (anti-kebocoran data).

Keluaran: `data/manifest.csv`, `data/crops/`, `data/dataset_stats.json`.

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Image

# Temukan root proyek (folder berisi configs/experiment.yaml), lalu jadikan cwd.
_cands = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(p for p in _cands if (p / "configs" / "experiment.yaml").exists())
os.chdir(ROOT)
for _p in (ROOT / "src", ROOT / "scripts"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from vehicle_lighting.config import load_config, resolve
cfg = load_config()
print("ROOT   :", ROOT)
print("config :", ROOT / "configs" / "experiment.yaml")
print("kelas  :", cfg["data"]["classes"])
print("model  :", cfg["models"])

## 2. Parameter dataset
Diambil dari `configs/experiment.yaml`; ubah di sana, bukan di notebook.

In [ ]:
params = dict(
    bdd_root=str(cfg["data"]["bdd100k"]["root"]),
    per_class=cfg["data"]["bdd100k"]["per_class"],
    min_box=cfg["data"]["bdd100k"]["min_box"],
    seed=cfg["project"]["seeds"][0],
    splits=["train", "val"],
)
params

## 3. Jalankan pra-proses

Menjalankan `scripts/bdd100k_to_dataset.py` (fungsi sama dengan versi CLI).
Pemindaian JSON bisa berlangsung lama. Untuk uji cepat, kecilkan `per_class`
dan batasi citra dengan `--limit-images`.

In [ ]:
import subprocess
cmd = [sys.executable, "scripts/bdd100k_to_dataset.py",
       "--bdd-root", params["bdd_root"],
       "--per-class", str(params["per_class"]),
       "--min-box", str(params["min_box"]),
       "--seed", str(params["seed"]),
       "--splits", *params["splits"]]
print(" ".join(cmd))
r = subprocess.run(cmd, text=True, capture_output=True)
print(r.stdout[-4000:])
if r.returncode:
    print("STDERR:\n", r.stderr[-2000:])

## 4. Periksa manifest

In [ ]:
import pandas as pd
from vehicle_lighting.data import check_leakage

df = pd.read_csv(resolve(cfg, cfg["data"]["manifest"]))
print("total crop:", len(df), "| jumlah grup (video):", df.group_id.nunique())
print("kebocoran data:", check_leakage(df) or "aman")
display(df.head())
display(df.pivot_table(index="label", columns="split", values="path",
                       aggfunc="count", fill_value=0))
display(df.pivot_table(index="label", columns="lighting", values="path",
                       aggfunc="count", fill_value=0))

**Gate tahap 1:** tiap kelas muncul di ketiga split dan tidak ada `group_id` yang
muncul di lebih dari satu split. Lanjut ke `02_verifikasi_pencahayaan.ipynb`.